In [1]:
!pip install --upgrade pip setuptools wheel
!pip install scikit-learn==1.4.2 --only-binary=:all:
!pip install xgboost

  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 204.3 MB/s  0:00:00
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (37.7 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 61.7 MB/s  0:00:02m0:00:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.4/303.4 MB 86.2 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [xgboost]m1/2 [xgboost]


In [27]:
import joblib
import numpy as np
import pandas as pd
import sklearn

from preprocessing import CreditScorePreprocessor
from train import CreditScoreTrainer
from evaluation import ModelEvaluator

print("Scikit-Learn Version:", sklearn.__version__)

Scikit-Learn Version: 1.4.2


In [ ]:
preprocessor = CreditScorePreprocessor()
df = preprocessor.clean_data("credit_score.csv")

print(df.shape)
df.head()

(25000, 21)


,Age,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,...,Outstanding_Debt,Credit_Utilization_Ratio,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score,Credit_History_Age_Months,Annual_Income_log
0,32.0,4875.125000,8.0,3.0,18.0,2.0,30.0,14.0,17.89,4.0,...,370.22,32.014182,Yes,81.822857,182.065510,High_spent_Medium_value_payments,473.624133,1,346.0,10.935363
1,39.0,NaN,9.0,7.0,24.0,5.0,56.0,25.0,NaN,8.0,...,2373.61,33.951720,Yes,258.848861,195.372215,High_spent_Small_value_payments,300.678924,0,198.0,11.037290
2,45.0,7471.013333,7.0,7.0,19.0,3.0,11.0,19.0,8.70,5.0,...,124.29,41.016763,Yes,129.723699,180.798742,High_spent_Medium_value_payments,686.578893,1,NaN,11.389414
3,32.0,468.770000,8.0,10.0,29.0,6.0,62.0,17.0,19.34,7.0,...,2924.76,34.203026,Yes,19.727923,39.937703,Low_spent_Medium_value_payments,267.211374,1,122.0,8.964726
4,18.0,NaN,3.0,4.0,8.0,3.0,25.0,20.0,11.27,2.0,...,1005.83,27.237916,No,227.241789,73.485708,High_spent_Large_value_payments,820.905003,1,392.0,11.588709


In [29]:
x_train, x_test, y_train, y_test = preprocessor.split_data(df)

print(x_train.shape)
print(x_test.shape)

(20000, 20)
(5000, 20)


In [30]:
preprocessor.save_split_data(x_train, x_test, y_train, y_test)

Train data saved to train/train.csv
Test data saved to test/test.csv


In [31]:
transformer = preprocessor.get_transformer(x_train)
transformer

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['Age', 'Monthly_Inhand_Salary',
                                  'Num_Bank_Accounts', 'Num_Credit_Card',
                                  'Interest_Rate', 'Num_of_Loan',
                                  'Delay_from_due_date',
                                  'Num_of_Delayed_Payment',
                                  'Changed_Credit_Limit',
                                  'Num_Credit_Inquiries', 'Outstanding_Debt',
                                  'Credit_Utilization_Ratio',
                                  'Total_EMI...
                                 ['Credit_Mix']),
                                ('payment_min',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OrdinalEncoder(categories=[['No',
                                                                              'Yes']]))]),
                                 ['Payment_of_Min_Amount']),
                                ('payment_behaviour',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'))]),
                                 ['Payment_Behaviour'])])

In [32]:
trainer = CreditScoreTrainer()
rf_model = trainer.train_random_forest(x_train, y_train, transformer)
xgb_model = trainer.train_xgboost(x_train, y_train, transformer)

Random Forest saved to artifacts/random_forest_pipeline.pkl
XGBoost saved to artifacts/xgboost_pipeline.pkl


In [33]:
evaluator = ModelEvaluator()
results = evaluator.run({"Random Forest": rf_model, "XGBoost": xgb_model}, x_test, y_test)

results

[{'Model': 'Random Forest',
  'Accuracy': 0.7308,
  'Weighted F1': np.float64(0.7318867226724849),
  'Test AUC': np.float64(0.8722207889426664)},
 {'Model': 'XGBoost',
  'Accuracy': 0.7156,
  'Weighted F1': np.float64(0.7150974570390725),
  'Test AUC': np.float64(0.860612905888907)}]

In [34]:
result_df = pd.DataFrame(results)

result_df

,Model,Accuracy,Weighted F1,Test AUC
0,Random Forest,0.7308,0.731887,0.872221
1,XGBoost,0.7156,0.715097,0.860613


In [35]:
best_model = max(results, key=lambda x: x["Weighted F1"])
best_model

{'Model': 'Random Forest',
 'Accuracy': 0.7308,
 'Weighted F1': np.float64(0.7318867226724849),
 'Test AUC': np.float64(0.8722207889426664)}

In [36]:
trainer.save_best_model(best_model["Model"])

Best model exported successfully
Source : artifacts/random_forest_pipeline.pkl
Destination : model/model_credit.joblib


PosixPath('model/model_credit.joblib')

In [37]:
model = joblib.load("model/model_credit.joblib")

model

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Age',
                                                   'Monthly_Inhand_Salary',
                                                   'Num_Bank_Accounts',
                                                   'Num_Credit_Card',
                                                   'Interest_Rate',
                                                   'Num_of_Loan',
                                                   'Delay_from_due_date',
                                                   'Num_of_Delayed_Payment',
                                                   'Changed_Credit_Limit',
                                                   'Num_Credit_Inquiries',
                                                   'Outstanding_Debt',
                                                   'Cre...
                                                                   OrdinalEncoder(categories=[['No',
                                                                                               'Yes']]))]),
                                                  ['Payment_of_Min_Amount']),
                                                 ('payment_behaviour',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore'))]),
                                                  ['Payment_Behaviour'])])),
                ('model',
                 RandomForestClassifier(class_weight='balanced',
                                        min_samples_split=5, n_estimators=500,
                                        random_state=42))])

In [38]:
sample = pd.DataFrame([
    {
        "Age":35,
        "Monthly_Inhand_Salary":4500,
        "Num_Bank_Accounts":4,
        "Num_Credit_Card":5,
        "Interest_Rate":8,
        "Num_of_Loan":2,
        "Delay_from_due_date":2,
        "Num_of_Delayed_Payment":1,
        "Changed_Credit_Limit":5,
        "Num_Credit_Inquiries":2,
        "Outstanding_Debt":1200,
        "Credit_Utilization_Ratio":25,
        "Total_EMI_per_month":180,
        "Amount_invested_monthly":700,
        "Monthly_Balance":3000,
        "Credit_Mix":"Good",
        "Payment_of_Min_Amount":"No",
        "Payment_Behaviour":"High_spent_Small_value_payments",
        "Credit_History_Age_Months":120,
        "Annual_Income_log":np.log1p(75000)
    }
])

prediction = model.predict(sample)
probability = model.predict_proba(sample)

label_map = {
    0:"Poor",
    1:"Standard",
    2:"Good"
}

print("Prediction :", label_map[prediction[0]])
print("Probability :", probability)

Prediction : Standard
Probability : [[0.16787206 0.45329248 0.37883546]]
